# FEATURE ENGINEERING WITH PYSPARK

## Bài toán

**Dự đoán giá bán bất động sản (`SalesClosePrice`)** từ các thuộc tính nhà ở tại St. Paul, Minnesota.

> Dataset có **5.000 dòng và 73 cột**.  

## Mục tiêu học tập

1. Đọc và kiểm tra dữ liệu với PySpark.
2. EDA bằng thống kê mô tả, correlation và sampling.
3. Làm sạch dữ liệu, xử lý outlier, duplicate và missing.
4. Scaling và transformation.
5. Join dữ liệu ngoài.
6. Sinh feature mới, time feature, lag feature và text feature.
7. Binarizing, bucketing và one-hot encoding.
8. Chuyển dữ liệu thành feature vector.
9. Train `RandomForestRegressor`.
10. Đánh giá bằng RMSE, MAE, R² và phân tích feature importance.

<a id="setup"></a>
# 1. Chuẩn bị môi trường

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    print("Đang cài PySpark ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])

print("Môi trường đã sẵn sàng.")

<a id="spark"></a>
## Khởi tạo SparkSession

In [ ]:
from pyspark.sql import SparkSession
import sys

spark = (
    SparkSession.builder
    .appName("FeatureEngineeringWithPySpark")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version :", spark.version)
print("Python version:", sys.version)

<a id="load"></a>
## Đọc dữ liệu

PySpark hỗ trợ nhiều định dạng:

```python
spark.read.json(...)
spark.read.csv(...)
spark.read.parquet(...)
```

In [ ]:
DATA_PATH = "2017_StPaul_MN_Real_Estate.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
)

df.show(5, truncate=False)

<a id="validate"></a>
## Kiểm tra dữ liệu sau khi load

- `df.count()` → số dòng.
- `df.columns` → danh sách cột.
- `len(df.columns)` → số cột.

In [ ]:
print("Số dòng:", df.count())
print("Số cột :", len(df.columns))
print(df.columns[:15])

### Hàm `check_load()`

In [ ]:
def check_load(df, expected_rows=None, expected_cols=None):
    actual_rows = df.count()
    actual_cols = len(df.columns)

    return {
        "actual_rows": actual_rows,
        "actual_cols": actual_cols,
        "rows_ok": expected_rows is None or actual_rows == expected_rows,
        "cols_ok": expected_cols is None or actual_cols == expected_cols
    }

check_load(df, expected_rows=5000, expected_cols=73)

<a id="schema"></a>
## 1. Kiểm tra schema và datatype

In [ ]:
df.printSchema()

In [ ]:
df.dtypes[:15]

### Kiểm tra datatype mong đợi

In [ ]:
expected_types = {
    "ASSESSMENTPENDING": "string",
    "AssessedValuation": "double",
    "AssociationFee": "bigint",
    "AssumableMortgage": "string",
    "SQFTBELOWGROUND": "bigint"
}

actual_types = dict(df.dtypes)

for col_name, expected_type in expected_types.items():
    print(
        f"{col_name:22s} expected={expected_type:8s} "
        f"actual={actual_types.get(col_name)}"
    )

<a id="eda"></a>
## 2. EDA – thống kê mô tả

In [ ]:
df.describe(
    ["LISTPRICE", "SalesClosePrice", "LivingArea", "YEARBUILT"]
).show()

In [ ]:
from pyspark.sql import functions as F

df.select(
    F.mean("SalesClosePrice").alias("mean_price"),
    F.stddev("SalesClosePrice").alias("std_price"),
    F.min("SalesClosePrice").alias("min_price"),
    F.max("SalesClosePrice").alias("max_price"),
    F.skewness("SalesClosePrice").alias("skew_price")
).show()

<a id="corr"></a>
### Correlation và Sampling

In [ ]:
print(
    "Corr(SalesClosePrice, YEARBUILT) =",
    df.corr("SalesClosePrice", "YEARBUILT")
)

print(
    "Corr(SalesClosePrice, LivingArea) =",
    df.corr("SalesClosePrice", "LivingArea")
)

### Tìm feature tương quan mạnh nhất

In [ ]:
numeric_candidates = [
    "FOUNDATIONSIZE", "DAYSONMARKET", "Fireplaces", "PDOM",
    "SQFTABOVEGROUND", "Taxes", "TAXWITHASSESSMENTS",
    "LivingArea", "YEARBUILT", "ACRES", "BathsFull",
    "BathsHalf", "BATHSTOTAL", "Bedrooms",
    "SQFTBELOWGROUND", "AssociationFee", "AssessedValuation"
]

corr_results = []

for c in numeric_candidates:
    if c in df.columns:
        try:
            value = df.corr(c, "SalesClosePrice")
            if value is not None:
                corr_results.append((c, value, abs(value)))
        except Exception:
            pass

corr_results = sorted(
    corr_results,
    key=lambda x: x[2],
    reverse=True
)

for c, value, _ in corr_results[:10]:
    print(f"{c:22s}: {value:.4f}")

### Sampling trước khi chuyển sang Pandas

**Không nên `toPandas()` toàn bộ Big Data.**  
Hãy sample trước.

In [ ]:
sample_df = (
    df.select("SalesClosePrice", "LivingArea")
      .sample(False, 0.2, 42)
)

print("Số dòng mẫu:", sample_df.count())

pandas_sample = sample_df.toPandas()
pandas_sample.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.scatter(
    pandas_sample["LivingArea"],
    pandas_sample["SalesClosePrice"],
    alpha=0.5
)
plt.xlabel("LivingArea")
plt.ylabel("SalesClosePrice")
plt.title("LivingArea vs SalesClosePrice")
plt.grid(alpha=0.3)
plt.show()

<a id="cleaning"></a>
## 2. Làm sạch dữ liệu

Các lỗi phổ biến:

- ghi sai;
- sự kiện hiếm;
- sai định dạng;
- duplicate;
- missing;
- dữ liệu không liên quan.

### Drop các cột không cần thiết

In [ ]:
cols_to_drop = [
    "StreetNumberNumeric",
    "LotSizeDimensions",
    "UNITNUMBER",
    "Class"
]

existing = [c for c in cols_to_drop if c in df.columns]
df_clean = df.drop(*existing)

print("Số cột sau khi drop:", len(df_clean.columns))

### Text filtering

In [ ]:
df_clean.select("PotentialShortSale").distinct().show(truncate=False)

df_clean = df_clean.where(
    (~F.col("PotentialShortSale").like("Not Disclosed"))
    | F.col("PotentialShortSale").isNull()
)

print("Số dòng sau filter:", df_clean.count())

### Duplicate

In [ ]:
before = df_clean.count()
df_clean = df_clean.dropDuplicates()
after = df_clean.count()

print("Trước:", before)
print("Sau  :", after)
print("Đã loại:", before - after)

<a id="outlier"></a>
## 3. Outlier Filtering μ ± 3σ

In [ ]:
row = df_clean.select(
    F.mean("SalesClosePrice").alias("mean"),
    F.stddev("SalesClosePrice").alias("std")
).first()

mean_val = row["mean"]
std_val = row["std"]

low_bound = mean_val - 3 * std_val
high_bound = mean_val + 3 * std_val

df_no_outlier = df_clean.where(
    (F.col("SalesClosePrice") >= low_bound)
    & (F.col("SalesClosePrice") <= high_bound)
)

print("Lower:", low_bound)
print("Upper:", high_bound)
print("Trước:", df_clean.count())
print("Sau  :", df_no_outlier.count())

> Quy tắc 3σ theo ví dụ. Với phân phối lệch mạnh có thể cân nhắc IQR hoặc percentile.

<a id="scaling"></a>
## 4. Scaling và Transformation

### Min-Max Scaling

In [ ]:
row = df_no_outlier.select(
    F.min("DAYSONMARKET").alias("min"),
    F.max("DAYSONMARKET").alias("max")
).first()

min_days, max_days = row["min"], row["max"]

df_scaled = df_no_outlier.withColumn(
    "scaled_days",
    (F.col("DAYSONMARKET") - F.lit(min_days))
    / (F.lit(max_days) - F.lit(min_days))
)

df_scaled.select("DAYSONMARKET", "scaled_days").show(5)

### Standardization Z-score

In [ ]:
row = df_scaled.select(
    F.mean("DAYSONMARKET").alias("mean"),
    F.stddev("DAYSONMARKET").alias("std")
).first()

mean_days, std_days = row["mean"], row["std"]

df_scaled = df_scaled.withColumn(
    "z_days",
    (F.col("DAYSONMARKET") - F.lit(mean_days))
    / F.lit(std_days)
)

df_scaled.select(
    F.mean("z_days").alias("mean_z"),
    F.stddev("z_days").alias("std_z")
).show()

### Log Scaling

In [ ]:
df_scaled = df_scaled.withColumn(
    "log_SalesClosePrice",
    F.log(F.col("SalesClosePrice"))
)

df_scaled.select(
    "SalesClosePrice",
    "log_SalesClosePrice"
).show(5)

### Min-Max cho nhiều cột

In [ ]:
def min_max_scale_columns(df, columns):
    out = df

    for c in columns:
        row = out.select(
            F.min(c).alias("min"),
            F.max(c).alias("max")
        ).first()

        min_v, max_v = row["min"], row["max"]

        if (
            min_v is not None
            and max_v is not None
            and max_v != min_v
        ):
            out = out.withColumn(
                f"{c}_minmax",
                (F.col(c) - F.lit(min_v))
                / (F.lit(max_v) - F.lit(min_v))
            )

    return out

df_scaled = min_max_scale_columns(
    df_scaled,
    ["FOUNDATIONSIZE", "DAYSONMARKET", "Fireplaces"]
)

<a id="missing"></a>
## 5. Missing Data

Trước khi impute, cần hiểu vì sao dữ liệu bị thiếu.

### Đếm missing

In [ ]:
missing_cols = [
    "ROOF", "RoomArea1", "RoomArea2",
    "APPLIANCES", "BASEMENT", "DAYSONMARKET"
]

for c in missing_cols:
    if c in df_scaled.columns:
        n = df_scaled.where(F.col(c).isNull()).count()
        print(f"{c:18s}: {n}")

### Mean imputation

In [ ]:
mean_dom = df_scaled.select(
    F.mean("DAYSONMARKET")
).first()[0]

df_imputed = df_scaled.fillna(
    {"DAYSONMARKET": mean_dom}
)

print(
    "Missing còn lại:",
    df_imputed.where(F.col("DAYSONMARKET").isNull()).count()
)

### Báo cáo missing

In [ ]:
def missing_report(df):
    total = df.count()
    result = []

    for c in df.columns:
        missing = df.where(F.col(c).isNull()).count()
        pct = missing / total if total else 0.0
        result.append((c, missing, pct))

    return sorted(result, key=lambda x: x[2], reverse=True)

report = missing_report(df_imputed)

for c, missing, pct in report[:15]:
    print(f"{c:25s} {missing:4d}  {pct:.1%}")

### Drop cột theo ngưỡng missing

In [ ]:
def drop_columns_by_missing_threshold(df, threshold=0.60):
    total = df.count()
    drop_cols = []

    for c in df.columns:
        missing = df.where(F.col(c).isNull()).count()
        if total and missing / total > threshold:
            drop_cols.append(c)

    return df.drop(*drop_cols), drop_cols

df_missing_clean, dropped_cols = (
    drop_columns_by_missing_threshold(
        df_imputed,
        threshold=0.60
    )
)

print("Các cột bị loại:")
print(dropped_cols)

<a id="join"></a>
## 6. Join dữ liệu ngoài

Ta tạo một bảng mortgage rate nhỏ để minh họa DataFrame Join và Spark SQL Join.

In [ ]:
mortgage_data = [
    ("2017-01", 4.15), ("2017-02", 4.17),
    ("2017-03", 4.20), ("2017-04", 4.05),
    ("2017-05", 4.01), ("2017-06", 3.90),
    ("2017-07", 3.97), ("2017-08", 3.88),
    ("2017-09", 3.81), ("2017-10", 3.90),
    ("2017-11", 3.92), ("2017-12", 3.95)
]

mortgage_df = spark.createDataFrame(
    mortgage_data,
    ["year_month", "mortgage_rate"]
)

mortgage_df.show()

In [ ]:
from pyspark.sql.functions import to_date, date_format

df_join = (
    df_missing_clean
    .withColumn("LISTDATE_DATE", to_date("LISTDATE"))
    .withColumn(
        "year_month",
        date_format("LISTDATE_DATE", "yyyy-MM")
    )
)

df_join = df_join.join(
    mortgage_df,
    on="year_month",
    how="left"
)

df_join.select(
    "LISTDATE",
    "year_month",
    "mortgage_rate"
).show(5)

### Spark SQL Join

In [ ]:
df_join.createOrReplaceTempView("real_estate")
mortgage_df.createOrReplaceTempView("mortgage")

sql_join = spark.sql("""
SELECT
    r.MLSID,
    r.LISTDATE,
    r.SalesClosePrice,
    r.year_month,
    m.mortgage_rate
FROM real_estate r
LEFT JOIN mortgage m
ON r.year_month = m.year_month
""")

sql_join.show(5)

<a id="generation"></a>
## 7. Feature Generation

Minh họa multiplication, summing, differencing và dividing.

### Tổng diện tích

In [ ]:
df_feat = df_join.withColumn(
    "SQFT_TOTAL",
    F.coalesce(F.col("SQFTABOVEGROUND"), F.lit(0))
    + F.coalesce(F.col("SQFTBELOWGROUND"), F.lit(0))
)

df_feat.select(
    "SQFTABOVEGROUND",
    "SQFTBELOWGROUND",
    "SQFT_TOTAL"
).show(5)

### Giá niêm yết trên mỗi sqft

In [ ]:
df_feat = df_feat.withColumn(
    "LISTING_PRICE_PER_SQFT",
    F.when(
        F.col("SQFT_TOTAL") > 0,
        F.col("LISTPRICE") / F.col("SQFT_TOTAL")
    )
)

df_feat.select(
    "LISTPRICE",
    "SQFT_TOTAL",
    "LISTING_PRICE_PER_SQFT"
).show(5)

### Tỷ lệ giá bán / giá niêm yết

In [ ]:
df_feat = df_feat.withColumn(
    "SALE_TO_LIST_RATIO",
    F.when(
        F.col("LISTPRICE") > 0,
        F.col("SalesClosePrice") / F.col("LISTPRICE")
    )
)

df_feat.select(
    "SalesClosePrice",
    "LISTPRICE",
    "SALE_TO_LIST_RATIO"
).show(5)

<a id="time"></a>
## 8. Time Features

In [ ]:
from pyspark.sql.functions import (
    year, month, dayofmonth, weekofyear, datediff
)

df_feat = (
    df_feat
    .withColumn("LIST_YEAR", year("LISTDATE_DATE"))
    .withColumn("LIST_MONTH", month("LISTDATE_DATE"))
    .withColumn("LIST_DAY", dayofmonth("LISTDATE_DATE"))
    .withColumn("LIST_WEEK", weekofyear("LISTDATE_DATE"))
)

df_feat.select(
    "LISTDATE_DATE",
    "LIST_YEAR",
    "LIST_MONTH",
    "LIST_DAY",
    "LIST_WEEK"
).show(5)

### Tính Days on Market từ hai ngày

In [ ]:
df_feat = df_feat.withColumn(
    "OFFMARKET_DATE",
    to_date("offmarketdate")
)

df_feat = df_feat.withColumn(
    "DAYS_ON_MARKET_FROM_DATES",
    datediff("OFFMARKET_DATE", "LISTDATE_DATE")
)

df_feat.select(
    "LISTDATE_DATE",
    "OFFMARKET_DATE",
    "DAYS_ON_MARKET_FROM_DATES"
).show(5)

<a id="lag"></a>
## 9. Lag Features

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

w = Window.orderBy("LISTDATE_DATE", "MLSID")

df_feat = df_feat.withColumn(
    "PREV_SALE_PRICE",
    lag("SalesClosePrice", 1).over(w)
)

df_feat.select(
    "LISTDATE_DATE",
    "SalesClosePrice",
    "PREV_SALE_PRICE"
).show(10)

> **Cảnh báo data leakage:** chỉ dùng lag feature nếu thông tin đó đã có ở thời điểm dự đoán.

<a id="text"></a>
## 10. Feature Extraction từ Text

Dùng `ROOF` để tạo:
- mái cũ hơn 8 năm;
- vật liệu mái.

In [ ]:
from pyspark.sql.functions import when, split

find_under_8 = F.col("ROOF").like(
    "%Age 8 Years or Less%"
)
find_over_8 = F.col("ROOF").like(
    "%Age Over 8 Years%"
)

df_feat = df_feat.withColumn(
    "old_roof",
    when(find_over_8, 1)
    .when(find_under_8, 0)
    .otherwise(None)
)

df_feat = df_feat.withColumn(
    "Roof_Material",
    split(F.col("ROOF"), ",").getItem(0)
)

df_feat.select(
    "ROOF",
    "old_roof",
    "Roof_Material"
).show(8, truncate=False)

### Explode & Pivot

In [ ]:
from pyspark.sql.functions import explode, lit, first

roof_tokens = (
    df_feat
    .select("MLSID", "ROOF")
    .where(F.col("ROOF").isNotNull())
    .withColumn("roof_list", split("ROOF", ", "))
    .withColumn("roof_token", explode("roof_list"))
    .withColumn("value", lit(1))
)

roof_pivot = (
    roof_tokens
    .groupBy("MLSID")
    .pivot("roof_token")
    .agg(first("value"))
    .fillna(0)
)

roof_pivot.show(3, truncate=False)

<a id="binning"></a>
## 11. Binarizing và Bucketing

### Binarizer – có fireplace hay không

In [ ]:
from pyspark.ml.feature import Binarizer

df_feat = df_feat.withColumn(
    "Fireplaces_double",
    F.col("Fireplaces").cast("double")
)

binarizer = Binarizer(
    threshold=0.0,
    inputCol="Fireplaces_double",
    outputCol="Has_Fireplace"
)

df_feat = binarizer.transform(df_feat)

df_feat.select(
    "Fireplaces",
    "Has_Fireplace"
).show(8)

### Bucketizer – nhóm số phòng tắm

In [ ]:
from pyspark.ml.feature import Bucketizer

df_feat = df_feat.withColumn(
    "BATHSTOTAL_double",
    F.col("BATHSTOTAL").cast("double")
)

splits = [
    float("-inf"), 1, 2, 3, 4, float("inf")
]

bucketizer = Bucketizer(
    splits=splits,
    inputCol="BATHSTOTAL_double",
    outputCol="Bath_Bucket"
)

df_feat = bucketizer.transform(df_feat)

df_feat.select(
    "BATHSTOTAL",
    "Bath_Bucket"
).show(8)

<a id="ohe"></a>
### One-Hot Encoding

Quy trình:

```text
String → StringIndexer → OneHotEncoder → Sparse Vector
```

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

city_indexer = StringIndexer(
    inputCol="City",
    outputCol="City_Index",
    handleInvalid="keep"
)

city_index_model = city_indexer.fit(df_feat)
df_encoded = city_index_model.transform(df_feat)

city_encoder = OneHotEncoder(
    inputCols=["City_Index"],
    outputCols=["City_Vec"]
)

city_encoder_model = city_encoder.fit(df_encoded)
df_encoded = city_encoder_model.transform(df_encoded)

df_encoded.select(
    "City",
    "City_Index",
    "City_Vec"
).show(8, truncate=False)

<a id="split"></a>
## 12. Train/Test Split theo thời gian

Không random split khi muốn mô phỏng dự đoán tương lai.

In [ ]:
date_quantiles = (
    df_encoded
    .select(
        F.col("LISTDATE_DATE")
         .cast("timestamp")
         .cast("long")
         .alias("ts")
    )
    .where(F.col("ts").isNotNull())
    .approxQuantile("ts", [0.8], 0.001)
)

split_ts = int(date_quantiles[0])

split_date = spark.sql(
    f"SELECT to_date(from_unixtime({split_ts})) AS d"
).first()["d"]

print("Split date:", split_date)

train_df = df_encoded.where(
    F.col("LISTDATE_DATE") < F.lit(split_date)
)

test_df = df_encoded.where(
    F.col("LISTDATE_DATE") >= F.lit(split_date)
)

print("Train:", train_df.count())
print("Test :", test_df.count())

<a id="vector"></a>
## 13. VectorAssembler

In [ ]:
from pyspark.ml.feature import VectorAssembler

numeric_features = [
    "LISTPRICE",
    "OriginalListPrice",
    "LivingArea",
    "SQFTABOVEGROUND",
    "SQFT_TOTAL",
    "YEARBUILT",
    "DAYSONMARKET",
    "Bedrooms",
    "BATHSTOTAL_double",
    "Has_Fireplace",
    "LIST_MONTH",
    "mortgage_rate"
]

numeric_features = [
    c for c in numeric_features
    if c in df_encoded.columns
]

fill_map = {
    c: 0.0
    for c in numeric_features
}

train_model_df = train_df.fillna(fill_map)
test_model_df = test_df.fillna(fill_map)

assembler_inputs = numeric_features + ["City_Vec"]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

train_ready = assembler.transform(train_model_df)
test_ready = assembler.transform(test_model_df)

train_ready.select(
    "SalesClosePrice",
    "features"
).show(5, truncate=False)

<a id="rf"></a>
## 14. Random Forest Regression

In [ ]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="SalesClosePrice",
    predictionCol="Prediction_Price",
    numTrees=100,
    maxDepth=8,
    seed=42
)

rf_model = rf.fit(train_ready)

predictions = rf_model.transform(test_ready)

predictions.select(
    "Prediction_Price",
    "SalesClosePrice"
).show(10)

<a id="eval"></a>
## 15. Đánh giá mô hình

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="SalesClosePrice",
    predictionCol="Prediction_Price"
)

rmse = evaluator.evaluate(
    predictions,
    {evaluator.metricName: "rmse"}
)

mae = evaluator.evaluate(
    predictions,
    {evaluator.metricName: "mae"}
)

r2 = evaluator.evaluate(
    predictions,
    {evaluator.metricName: "r2"}
)

print(f"RMSE = {rmse:,.2f}")
print(f"MAE  = {mae:,.2f}")
print(f"R²   = {r2:.4f}")

- **RMSE**: nhạy với lỗi lớn.
- **MAE**: dễ diễn giải theo đơn vị giá.
- **R²**: mức độ giải thích biến thiên của target.

<a id="importance"></a>
## 16. Feature Importance

In [ ]:
import pandas as pd

importance = rf_model.featureImportances.toArray()

metadata = train_ready.schema["features"].metadata
attrs = metadata["ml_attr"]["attrs"]

feature_names = []

for group in ["numeric", "binary", "nominal"]:
    for item in attrs.get(group, []):
        feature_names.append(
            (item["idx"], item["name"])
        )

feature_names = [
    name
    for idx, name
    in sorted(feature_names, key=lambda x: x[0])
]

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values(
    "importance",
    ascending=False
)

importance_df.head(15)

In [ ]:
top_imp = importance_df.head(15).sort_values(
    "importance"
)

plt.figure(figsize=(8, 6))
plt.barh(
    top_imp["feature"],
    top_imp["importance"]
)
plt.xlabel("Feature Importance")
plt.title("Top Feature Importance – Random Forest")
plt.tight_layout()
plt.show()

> Feature importance không chứng minh quan hệ nhân quả.

<a id="save"></a>
## 17. Lưu và tải model

In [ ]:
MODEL_PATH = "rfr_real_estate_model"

# Lưu model:
# rf_model.write().overwrite().save(MODEL_PATH)

In [ ]:
# Tải model:
#
# from pyspark.ml.regression import RandomForestRegressionModel
# loaded_model = RandomForestRegressionModel.load(MODEL_PATH)

### Mở rộng – Pipeline

Trong dự án thật nên gom preprocessing và model vào `Pipeline`.

In [ ]:
from pyspark.ml import Pipeline

# Ví dụ cấu trúc:
#
# pipeline = Pipeline(stages=[
#     city_indexer,
#     city_encoder,
#     assembler,
#     rf
# ])
#
# pipeline_model = pipeline.fit(train_df)
# pipeline_predictions = pipeline_model.transform(test_df)

<a id="exercise"></a>
## Bài tập mở rộng

### Bài 1 – Data Validation
Viết hàm kiểm tra số dòng, số cột, target và tỷ lệ missing của target.

### Bài 2 – EDA
Tìm 10 feature số có correlation tuyệt đối cao nhất với `SalesClosePrice`.

### Bài 3 – Outlier
So sánh quy tắc 3σ và IQR.

### Bài 4 – Scaling
Min-Max và Z-score cho `LISTPRICE`, `LivingArea`, `DAYSONMARKET`.

### Bài 5 – Missing
Tạo bảng số missing và % missing cho tất cả cột.

### Bài 6 – Feature Generation
Tạo:
- tuổi ngôi nhà;
- số phòng / 1.000 sqft;
- tỷ lệ thuế / giá niêm yết.

### Bài 7 – Time Feature
Tạo quarter, day of week và season.

### Bài 8 – Text Feature
Trích xuất feature từ `ROOF`, `BASEMENT`, `GarageDescription`.

### Bài 9 – Encoding
Encode `City` và `STYLE`.

### Bài 10 – Modeling
So sánh `LinearRegression`, `RandomForestRegressor`, `GBTRegressor`.

<a id="project"></a>
## 26. Bài tập nâng cao

### Pipeline tối thiểu

```text
Raw Data
  ↓
Validation
  ↓
EDA
  ↓
Cleaning / Missing / Outlier
  ↓
Feature Engineering
  ↓
Encoding
  ↓
VectorAssembler
  ↓
Train/Test Split
  ↓
Regression
  ↓
Evaluation
  ↓
Interpretation
```

## Yêu cầu

Xây dựng pipeline hoàn chỉnh từ dữ liệu thô đến mô hình hồi quy.
1. Load và validate dữ liệu.
2. EDA và kiểm tra missing/outlier.
3. Làm sạch dữ liệu.
4. Tạo ít nhất 5 feature mới.
5. Encode categorical features.
6. Tạo feature vector.
7. Chia train/test hợp lý.
8. So sánh ít nhất 2 mô hình.
9. Báo cáo RMSE, MAE và R².
10. Phân tích Top-10 feature quan trọng.
11. Thảo luận data leakage, missing bias và high-cardinality.

<a id="summary"></a>
## Tổng kết

```text
Understand Problem
      ↓
Load & Validate
      ↓
EDA
      ↓
Clean / Missing / Outlier
      ↓
Scale / Transform
      ↓
Join External Data
      ↓
Generate / Extract Features
      ↓
Encode
      ↓
VectorAssembler
      ↓
Train / Predict
      ↓
Evaluate / Interpret
```

## Ghi nhớ

1. Feature Engineering phải gắn với bài toán.
2. Không phải feature tạo được đều nên dùng.
3. Sample trước khi `toPandas()`.
4. Hiểu nguyên nhân missing trước khi impute.
5. Tránh data leakage.
6. Preprocessing train/test phải nhất quán.
7. Feature importance không đồng nghĩa với causal importance.